# Oasis Infotech Internship — Level 1 Task 4  
# Sentiment Analysis using Machine Learning

## Objective  
Build a machine learning model that classifies text sentiment into **positive**, **negative**, or **neutral**.

## Tech Stack  
Python, pandas, scikit-learn, NLTK, matplotlib, seaborn, WordCloud, Jupyter Notebook

## 1. Import Required Libraries

This section imports all required Python libraries for data handling, text preprocessing, visualisation, model building, and evaluation.


In [ ]:

import os
import re
import string
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Install missing packages automatically if needed
import sys
import subprocess

def install_if_missing(package_name, import_name=None):
    import_name = import_name or package_name
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

install_if_missing("nltk")
install_if_missing("wordcloud")
install_if_missing("scikit-learn", "sklearn")

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer

from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

nltk.download("stopwords", quiet=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")


ModuleNotFoundError: No module named 'sklearn'

## 2. Load Dataset and Initial Inspection

The dataset is loaded and inspected using:
- Shape
- Column names
- Data types
- Missing values
- First few records

> **Note:** Rename your downloaded file to one of these names if needed:  
> `twitter_training.csv`, `sentiment_dataset.csv`, `train.csv`, or `Twitter_Data.csv`.


In [ ]:

# Change this filename if your dataset has a different name
DATA_PATH = "twitter_training.csv"

possible_paths = [
    DATA_PATH,
    "sentiment_dataset.csv",
    "train.csv",
    "Twitter_Data.csv",
    "twitter_validation.csv"
]

existing_paths = [p for p in possible_paths if os.path.exists(p)]

if not existing_paths:
    raise FileNotFoundError(
        "No dataset file found. Put your CSV file in the same folder as this notebook "
        "and update DATA_PATH with the correct filename."
    )

file_path = existing_paths[0]
print("Using dataset:", file_path)

def read_csv_safely(path, header="infer"):
    try:
        return pd.read_csv(path, header=header)
    except UnicodeDecodeError:
        return pd.read_csv(path, header=header, encoding="latin1")

df_raw = read_csv_safely(file_path)
print("Initial shape:", df_raw.shape)
display(df_raw.head())

print("\nColumn data types:")
display(df_raw.dtypes)

print("\nMissing values:")
display(df_raw.isnull().sum())


### Observation  
Write your observation here after running the cell.

Example:
- The dataset contains `_____` rows and `_____` columns.
- The main text column appears to be `_____`.
- The sentiment column appears to be `_____`.
- Missing values are present in `_____` columns.


## 3. Standardise Text and Sentiment Columns

Different Kaggle datasets use different column names.  
This section tries to automatically detect and standardise them into:

- `text`
- `sentiment`

For the Twitter Entity Sentiment dataset, the file may not have headers, so this notebook also handles that format.


In [ ]:

def standardise_sentiment_dataset(path):
    temp = read_csv_safely(path)

    # Common text and sentiment column names
    text_candidates = [
        "text", "tweet", "tweet_text", "content", "review", "comment",
        "message", "clean_text", "Text", "Tweet", "Review", "Comment", "Sentence"
    ]
    sentiment_candidates = [
        "sentiment", "label", "class", "target", "polarity",
        "Sentiment", "Label", "Class", "Target", "Polarity"
    ]

    cols = list(temp.columns)
    text_col = next((c for c in cols if c in text_candidates), None)
    sentiment_col = next((c for c in cols if c in sentiment_candidates), None)

    # Handle Twitter Entity Sentiment Analysis format:
    # id, entity, sentiment, text
    if text_col is None or sentiment_col is None:
        temp_no_header = read_csv_safely(path, header=None)
        if temp_no_header.shape[1] >= 4:
            temp_no_header = temp_no_header.iloc[:, :4]
            temp_no_header.columns = ["tweet_id", "entity", "sentiment", "text"]
            temp = temp_no_header
            text_col = "text"
            sentiment_col = "sentiment"
        elif temp_no_header.shape[1] >= 2:
            temp_no_header = temp_no_header.iloc[:, :2]
            temp_no_header.columns = ["sentiment", "text"]
            temp = temp_no_header
            text_col = "text"
            sentiment_col = "sentiment"
        else:
            raise ValueError("Could not identify text and sentiment columns. Please rename them manually.")

    data = temp[[text_col, sentiment_col]].copy()
    data.columns = ["text", "sentiment"]

    # Drop blank rows
    data = data.dropna(subset=["text", "sentiment"])
    data["text"] = data["text"].astype(str)

    # Standardise sentiment labels
    s = data["sentiment"]

    # Numeric sentiment handling
    numeric_s = pd.to_numeric(s, errors="coerce")
    if numeric_s.notna().mean() > 0.9:
        unique_values = set(numeric_s.dropna().astype(int).unique())

        if unique_values.issubset({0, 2, 4}):
            mapping = {0: "negative", 2: "neutral", 4: "positive"}
        elif unique_values.issubset({-1, 0, 1}):
            mapping = {-1: "negative", 0: "neutral", 1: "positive"}
        elif unique_values.issubset({0, 1, 2}):
            mapping = {0: "negative", 1: "neutral", 2: "positive"}
        else:
            mapping = {}

        data["sentiment"] = numeric_s.astype("Int64").map(mapping)
    else:
        data["sentiment"] = (
            s.astype(str)
            .str.strip()
            .str.lower()
            .replace({
                "pos": "positive",
                "neg": "negative",
                "neu": "neutral",
                "irrelevant": "neutral",
                "not relevant": "neutral"
            })
        )

    # Keep only required classes
    data = data[data["sentiment"].isin(["positive", "negative", "neutral"])]

    # Remove duplicate text rows
    data = data.drop_duplicates(subset=["text"]).reset_index(drop=True)

    return data

df = standardise_sentiment_dataset(file_path)

print("Cleaned shape:", df.shape)
display(df.head())

print("\nFinal sentiment classes:")
display(df["sentiment"].value_counts())


### Observation  
Write your observation here.

Example:
- The final dataset contains only three sentiment classes: positive, negative, and neutral.
- If the original dataset contained `Irrelevant`, it has been treated as neutral because it does not show a clear positive or negative opinion.


## 4. Class Distribution Visualisation

Before training a model, we check whether the dataset is balanced or imbalanced.

A balanced dataset has roughly similar numbers of positive, negative, and neutral examples.  
An imbalanced dataset may cause the model to become biased toward the majority class.


In [ ]:

sentiment_counts = df["sentiment"].value_counts()

display(sentiment_counts)

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="sentiment", order=sentiment_counts.index)
plt.title("Sentiment Distribution in Dataset")
plt.xlabel("Sentiment")
plt.ylabel("Number of Text Samples")
plt.show()


### Observation  
Write your observation here.

Example:
- The most common sentiment is `_____`.
- The least common sentiment is `_____`.
- If one class is much larger than others, the model may perform better on that class.


## 5. Text Preprocessing Pipeline

Text data is unstructured, so it needs to be cleaned before machine learning.

The preprocessing steps used here are:

1. Convert text to lowercase  
2. Remove URLs  
3. Remove user mentions  
4. Remove punctuation and numbers  
5. Tokenise text into words  
6. Remove stopwords like `the`, `is`, `and`  
7. Apply stemming to reduce words to their root form  

Example:  
`Loved the product!!!` → `love product`


In [ ]:

stop_words = set(stopwords.words("english"))
tokenizer = RegexpTokenizer(r"[a-z]+")
stemmer = PorterStemmer()

def preprocess_text(text, use_stemming=True):
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove mentions but keep hashtag words by only removing '#'
    text = re.sub(r"@\w+", " ", text)
    text = text.replace("#", " ")

    # Remove punctuation, digits, and special characters
    text = re.sub(r"[^a-z\s]", " ", text)

    # Tokenisation
    tokens = tokenizer.tokenize(text)

    # Stopword removal and short-word filtering
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    # Optional stemming
    if use_stemming:
        tokens = [stemmer.stem(word) for word in tokens]

    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess_text)

# Remove rows where preprocessing made text empty
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

display(df[["text", "clean_text", "sentiment"]].head(10))


### Observation  
Write your observation here.

Example:
- The text is now cleaner and contains fewer unnecessary words.
- Punctuation, links, and stopwords have been removed.
- This helps the model focus on meaningful words related to sentiment.


## 6. Additional Visualisation: Text Length by Sentiment

This visualisation checks whether some sentiment classes tend to have longer or shorter text.

This can reveal a non-obvious insight. For example, negative reviews may be longer because customers explain complaints in more detail.


In [ ]:

df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))

display(df.groupby("sentiment")["word_count"].describe())

plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="sentiment", y="word_count")
plt.title("Text Length Distribution by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Word Count After Cleaning")
plt.show()


### Observation  
Write your observation here.

Example:
- `_____` sentiment texts appear longer on average.
- Very short texts may be harder for the model to classify because they provide less context.


## 7. WordCloud for Each Sentiment Class

WordClouds show the most frequent words in each sentiment class.

This helps us understand what type of words are commonly associated with positive, negative, and neutral opinions.


In [ ]:

for sentiment in ["positive", "negative", "neutral"]:
    text_data = " ".join(df[df["sentiment"] == sentiment]["clean_text"])

    if text_data.strip():
        wordcloud = WordCloud(
            width=900,
            height=450,
            background_color="white",
            max_words=120
        ).generate(text_data)

        plt.figure(figsize=(11, 5))
        plt.imshow(wordcloud, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"WordCloud for {sentiment.capitalize()} Sentiment")
        plt.show()
    else:
        print(f"No text available for {sentiment} sentiment.")


### Observation  
Write your observation here.

Example:
- Positive texts commonly include words like `_____`.
- Negative texts commonly include words like `_____`.
- Neutral texts commonly include words like `_____`.


## 8. Feature Extraction using TF-IDF Vectorizer

Machine learning models cannot directly understand raw text.  
So, we convert text into numerical features.

### What is TF-IDF?

**TF-IDF** stands for **Term Frequency–Inverse Document Frequency**.

It gives importance to words that are frequent in a document but not common across all documents.

Example:
- Common words like `good` or `product` may appear often.
- More specific words like `excellent`, `broken`, `delay`, or `refund` may carry stronger sentiment meaning.

TF-IDF helps convert cleaned text into a numerical matrix that can be used by machine learning classifiers.


In [ ]:

X = df[["text", "clean_text"]]
y = df["sentiment"]

# Stratify only if each class has enough samples
class_counts = y.value_counts()
stratify_target = y if class_counts.min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=stratify_target
)

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train["clean_text"])
X_test_tfidf = tfidf.transform(X_test["clean_text"])

print("Training samples:", X_train_tfidf.shape[0])
print("Testing samples:", X_test_tfidf.shape[0])
print("Number of TF-IDF features:", X_train_tfidf.shape[1])


### Observation  
Write your observation here.

Example:
- The dataset was split into 80% training and 20% testing data.
- TF-IDF created `_____` numerical features from the text data.


## 9. Train Machine Learning Models

We train two classifiers:

1. **Naive Bayes**  
   A simple and fast model commonly used for text classification.

2. **Logistic Regression**  
   A strong linear classification model that often performs well with TF-IDF features.


In [ ]:

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )
}

results = {}
predictions = {}

for model_name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    predictions[model_name] = y_pred

    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    results[model_name] = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    }

    print("=" * 70)
    print(model_name)
    print("=" * 70)
    print("Accuracy:", round(accuracy, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))


### Observation  
Write your observation here.

Example:
- Naive Bayes achieved an accuracy of `_____`.
- Logistic Regression achieved an accuracy of `_____`.
- Compare precision, recall, and F1-score to decide the better model.


## 10. Confusion Matrix for Each Model

A confusion matrix shows how many samples were correctly and incorrectly classified for each sentiment class.

Rows represent actual classes.  
Columns represent predicted classes.


In [ ]:

labels = sorted(y.unique())

for model_name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted Sentiment")
    plt.ylabel("Actual Sentiment")
    plt.show()


### Observation  
Write your observation here.

Example:
- The model mostly confuses `_____` with `_____`.
- Neutral sentiment may be difficult to classify because neutral texts often contain fewer emotional words.


## 11. Model Comparison

We compare both models using accuracy, precision, recall, and F1-score.

For sentiment analysis, **F1-score** is very important because it balances precision and recall.


In [ ]:

results_df = pd.DataFrame(results).T.sort_values("F1-score", ascending=False)
display(results_df)

results_df.plot(kind="bar", figsize=(10, 6))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()

best_model_name = results_df.index[0]
best_model = models[best_model_name]

print("Best model based on macro F1-score:", best_model_name)


### Observation  
Write your observation here.

Example:
- The best performing model is `_____`.
- It performed better because it achieved the highest macro F1-score.


## 12. Error Analysis

Error analysis helps us understand where the model fails.

We show 5 misclassified examples from the best model and add possible reasons.


In [ ]:

best_predictions = predictions[best_model_name]

error_df = pd.DataFrame({
    "original_text": X_test["text"].values,
    "clean_text": X_test["clean_text"].values,
    "actual_sentiment": y_test.values,
    "predicted_sentiment": best_predictions
})

misclassified = error_df[
    error_df["actual_sentiment"] != error_df["predicted_sentiment"]
].copy()

def guess_error_reason(text):
    lower_text = str(text).lower()
    word_count = len(lower_text.split())

    if word_count <= 4:
        return "Very short text; not enough context for the model."
    if any(word in lower_text for word in ["but", "however", "although", "though"]):
        return "Mixed sentiment; the sentence may contain both positive and negative signals."
    if "!" in lower_text or "?" in lower_text:
        return "Emotional or conversational wording may be hard to interpret."
    if any(word in lower_text for word in ["lol", "lmao", "sarcasm", "yeah right"]):
        return "Possible slang or sarcasm."
    return "May contain context-specific words, slang, or weak sentiment signals."

misclassified["possible_reason"] = misclassified["original_text"].apply(guess_error_reason)

print("Total misclassified examples:", len(misclassified))
display(misclassified.head(5))


### Error Analysis Discussion  
Write your discussion here.

Example:
- Some texts are misclassified because they are very short.
- Some texts contain mixed sentiment, such as praise and complaint in the same sentence.
- Some texts may include sarcasm, slang, or topic-specific words that simple TF-IDF models cannot fully understand.


## 13. Test the Model on Custom Sentences

Use this section to test the best model on your own examples.


In [ ]:

custom_texts = [
    "I absolutely love this product, it works perfectly!",
    "The service was terrible and I want a refund.",
    "The package arrived yesterday and the color is blue.",
    "The app is okay but it crashes sometimes.",
    "Amazing experience, I will recommend it to everyone."
]

custom_clean = [preprocess_text(text) for text in custom_texts]
custom_tfidf = tfidf.transform(custom_clean)
custom_predictions = best_model.predict(custom_tfidf)

custom_results = pd.DataFrame({
    "Text": custom_texts,
    "Cleaned Text": custom_clean,
    "Predicted Sentiment": custom_predictions
})

display(custom_results)


## 14. Conclusion

### Best Model  
After comparing Naive Bayes and Logistic Regression, the best model based on macro F1-score is:

`Run the previous cells to get the best model name automatically.`

### Real-World Application  
This sentiment analysis model can be used for:

1. **Customer feedback analysis**  
   Companies can automatically classify reviews as positive, negative, or neutral.

2. **Brand monitoring**  
   Businesses can track public opinion about their brand on social media.

3. **Product improvement**  
   Negative feedback can be analysed to identify common customer pain points.

### Final Recommendations  

1. Focus on the most common negative keywords from the negative WordCloud to understand recurring issues.  
2. Monitor neutral feedback separately because it may contain useful product information without emotional language.  
3. Use the best-performing model for automated feedback classification, but manually review low-confidence or unclear predictions.  
4. Improve future performance by collecting more balanced data and using advanced models like SVM, LSTM, or BERT.


## Checklist Mapping

| Requirement | Covered? |
|---|---|
| Load dataset and inspect class distribution | Yes |
| Lowercase, punctuation removal, stopword removal, tokenisation, stemming | Yes |
| TF-IDF Vectorizer with explanation | Yes |
| 80/20 train-test split | Yes |
| Naive Bayes + Logistic Regression | Yes |
| Accuracy, precision, recall, F1-score | Yes |
| Confusion matrix for each model | Yes |
| Sentiment distribution bar chart | Yes |
| WordCloud for each sentiment class | Yes |
| Error analysis with 5 misclassified examples | Yes |
| Conclusion with best model and application | Yes |
